## Evaluate agents_agents_lab-product-product_agent


This is the notebook version of your Playground session that used [Agent Evaluation](https://docs.databricks.com/generative-ai/agent-evaluation/index.html) to measure the quality of your Agent.  It includes:
1. The requests you entered
2. The code used to run Agent Evaluation on those requests.

You can use this notebook to edit the requests in your evaluation dataset and to run evaluation on different versions of your agent, leveraging mlflow to track the computed quality metrics.

In [0]:
%pip install databricks-agents
dbutils.library.restartPython()

# Agent endpoint

Below, `endpoint` is the Model Serving endpoint you used in Playground. `agent` is a function that calls into the endpoint to get the response.

*Note: If you want to run these requests against another Agents, `agent` can be a serving endpoint (`"endpoints:/..."`), a UC model (`"models:/..."`), an mlflow-registered model (`"runs:/..."`), or simply a function that calls into the agent. You can read more about these options in the [Agent Evaluation documentation](https://docs.databricks.com/generative-ai/agent-evaluation/evaluate-agent.html#example-agent-evaluation-runs-application).*

In [0]:
endpoint="agents_agents_lab-product-product_agent"

import mlflow.deployments

def agent(input):
  client = mlflow.deployments.get_deploy_client("databricks")
  
  return client.predict(endpoint=endpoint, inputs=input)

# Requests

Agent Evaluation can be used to either:
1. Evaluate each single turn of conversation independently
2. Evaluate the final turn of a multiple-turn conversation

Using your requests, we show you both options below.  It is feasible to create evaluation datasets that mix both single- and multi-turn questions.  See [schema for request](https://docs.databricks.com/generative-ai/agent-evaluation/evaluation-set.html#schema-for-request).

*Note: Agent Evaluation optionally allows you to provide a ground truth (or reference answer) to your questions.  If you do so, Agent Evaluation is able to assess the correctness (accuracy) of your Agent.  You can read more about the [schema of the evaluation dataset](https://docs.databricks.com/generative-ai/agent-evaluation/evaluation-set.html#evaluation-set-schema).*

## Single turn
The following snippet turns your requests into an evaluation dataset with single-turn questions.  That is, each question will be sent to the Agent independently of one another.

In [0]:
import pandas as pd

examples =  [
    {
        "request": "WHAT IS TODAY DATE",
        "expected_response": None  # Optional, fill in if you have a reference answer
    },
    {
        "request": "Requesting return for Wireless Headphones due to malfunction.",
        "expected_response": None  # Optional, fill in if you have a reference answer
    },
    # Add more questions here as needed, e.g.
    # {
    #     "request": "What is the capital of France?",
    #     "expected_response": "Paris"
    # },
]
eval_dataset = pd.DataFrame(examples)
display(eval_dataset)

In [0]:
import mlflow
result = mlflow.evaluate(
    agent,
    data=eval_dataset,              # Your evaluation dataset
    model_type="databricks-agent",  # Enable Mosaic AI Agent Evaluation
)

# Review the evaluation results in the MLFLow UI (see console output), or access them in place:
display(result.tables['eval_results'])

## Multi-turn questions

The following snippet turns your requests into an evaluation dataset with multi-turn questions.  That is, Agent Evaluation will send the entire conversation to your Agent and *only* evaluate the Agent's response to the last request.

In [0]:

examples_from_chat = [
    {
        "request": {
            "messages": [
                {
                    "role": "user",
                    "content": "WHAT IS TODAY DATE"
                }
            ]
        },
        "expected_response": None
    },
    {
        "request": {
            "messages": [
                {
                    "role": "user",
                    "content": "WHAT IS TODAY DATE"
                },
                {
                    "role": "assistant",
                    "content": "<tool_call>{\n  \"id\": \"call_ada30337-3479-49ae-a0af-a0f27bfeee8c\",\n  \"name\": \"agents_lab__product__get_todays_date\",\n  \"arguments\": \"{}\"\n}</tool_call>\n\n<tool_call_result>{\n  \"id\": \"call_ada30337-3479-49ae-a0af-a0f27bfeee8c\",\n  \"content\": \"{\\\"error\\\": \\\"ServiceErrorCode.BAD_REQUEST: [INSUFFICIENT_PERMISSIONS] Insufficient privileges:\\\\nUser does not have EXECUTE on Routine or Model 'agents_lab.product.get_todays_date'. SQLSTATE: 42501\\\"}\"\n}</tool_call_result>\n\nI'm sorry, but I am unable to retrieve today's date due to insufficient permissions.\n\n"
                },
                {
                    "role": "user",
                    "content": "Requesting return for Wireless Headphones due to malfunction."
                }
            ]
        },
        "expected_response": None
    }
]
eval_dataset_from_chat = pd.DataFrame(examples_from_chat)
display(eval_dataset_from_chat)

In [0]:
result_from_chat = mlflow.evaluate(
    agent,
    data=eval_dataset_from_chat,    # Your evaluation dataset
    model_type="databricks-agent",  # Enable Mosaic AI Agent Evaluation
)

# Review the evaluation results in the MLFLow UI (see console output), or access them in place:
display(result_from_chat.tables['eval_results'])